# Week 8 dragon demo

It’s 12:00. Blue can reach dragon a little sooner than Red, and Red’s mid tower is damaged. Blue can contest the dragon or give it up for a chance to take the tower.

In [1]:
from src.game.engine import load_scenario, resolve_action
from src.modeling.predict import predict_blue_win

scenario = load_scenario()

starting_state = {
    "elapsed_seconds": 720,
    "blue": {
        "total_gold": 22000,
        "kills": 9,
        "towers_destroyed": 1,
        "dragons_secured": 1,
        "barons_secured": 0,
    },
    "red": {
        "total_gold": 20000,
        "kills": 7,
        "towers_destroyed": 0,
        "dragons_secured": 0,
        "barons_secured": 0,
    },
}

starting_probability = predict_blue_win(starting_state)
print(f"Starting Blue win estimate: {starting_probability:.2%}")

Starting Blue win estimate: 63.43%


At 12:00, before the player chooses an action, B’s model estimates that Blue has a 63.43% chance of winning the whole match.

## Try both choices

Contest dragon gives the dragon to whichever team wins the contest. Push mid gives Red the dragon either way, and Blue gets the tower if the push succeeds. The action chances are demo rules. After each decision, B's model estimates Blue's chance of winning the match from the new scoreboard.

Each run starts at 12:00. Seed 1 gives a success for both actions, and seed 2 gives a failure.

In [2]:
results = {}

for action_id, action in scenario["actions"].items():
    print(f"{action['label']} ({action['success_chance']:.0%} action success chance)")
    for seed in (1, 2):
        result = resolve_action(starting_state, action_id, seed)
        state = result["state"]
        minutes, seconds = divmod(state["elapsed_seconds"], 60)
        probability = predict_blue_win(state)
        results[action_id, seed] = (result, probability)

        print(
            f"  Seed {seed} | {result['outcome']} | {minutes}:{seconds:02d} | "
            f"Dragons (Blue–Red) {state['blue']['dragons_secured']}–"
            f"{state['red']['dragons_secured']} | "
            f"Blue towers {state['blue']['towers_destroyed']} | "
            f"Blue win estimate {probability:.2%}"
        )

Contest dragon (60% action success chance)
  Seed 1 | success | 12:30 | Dragons (Blue–Red) 2–0 | Blue towers 1 | Blue win estimate 64.65%
  Seed 2 | failure | 12:35 | Dragons (Blue–Red) 1–1 | Blue towers 1 | Blue win estimate 62.18%
Push mid tower (75% action success chance)
  Seed 1 | success | 12:25 | Dragons (Blue–Red) 1–1 | Blue towers 2 | Blue win estimate 55.50%
  Seed 2 | failure | 12:30 | Dragons (Blue–Red) 1–1 | Blue towers 1 | Blue win estimate 62.18%


One result to check with B is the mid-tower push. Taking mid tower adds one Blue tower, yet the model's estimate drops to 55.50%, compared with 62.18% when Blue backs away. The saved model gives the tower difference a negative weight. 

## Draft decision log

In [3]:
import json
from pprint import pprint

from src.features.state import FEATURES
from src.modeling.predict import DEFAULT_MODEL

model = json.loads(DEFAULT_MODEL.read_text())

result, probability = results["contest_dragon", 1]
decision_log = {
    "session_id": "week8-demo",
    "round": 1,
    "action_id": result["action_id"],
    "seed": result["seed"],
    "success_chance": result["success_chance"],
    "random_draw": result["draw"],
    "outcome": result["outcome"],
    "before_state": starting_state,
    "after_state": result["state"],
    "blue_win_before": starting_probability,
    "blue_win_after": probability,
    "scenario_id": result["scenario_id"],
    "scenario_version": result["scenario_version"],
    "simulator_version": result["simulator_version"],
    "model_format_version": model["version"],
    "feature_version": None,  # waiting on B
    "feature_order": list(FEATURES),
}

pprint(decision_log, sort_dicts=False)

{'session_id': 'week8-demo',
 'round': 1,
 'action_id': 'contest_dragon',
 'seed': 1,
 'success_chance': 0.6,
 'random_draw': 0.13436424411240122,
 'outcome': 'success',
 'before_state': {'elapsed_seconds': 720,
                  'blue': {'total_gold': 22000,
                           'kills': 9,
                           'towers_destroyed': 1,
                           'dragons_secured': 1,
                           'barons_secured': 0},
                  'red': {'total_gold': 20000,
                          'kills': 7,
                          'towers_destroyed': 0,
                          'dragons_secured': 0,
                          'barons_secured': 0}},
 'after_state': {'elapsed_seconds': 750,
                 'blue': {'total_gold': 22000,
                          'kills': 9,
                          'towers_destroyed': 1,
                          'dragons_secured': 2,
                          'barons_secured': 0},
                 'red': {'total_gold': 20000,
     